In [1]:
# Vector addition with pycuda(GPU) and numba(CPU), optimized in/out.
# Import and initialize pycuda, numpy, numba, and time
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import numpy as np
import time 
from numba import njit, prange

In [2]:
#################### Array size
N = 10000000

In [3]:
# Create some space on CPU/HOST (random 32-bit ints)
a_cpu = np.random.uniform(1.0, 100.0, size=(N)).astype(np.uint32) 
b_cpu = np.random.uniform(1.0, 100.0, size=(N)).astype(np.uint32)
c_cpu = np.zeros(N, np.uint32)

In [4]:
#################### Write GPU kernel
module = SourceModule(""" 
    __global__ void addition(int* a_gpu, int* b_gpu, int* c_gpu, int N){
        // Global thread indices
        int id = threadIdx.x + blockIdx.x * blockDim.x;
        if(id < N){
            c_gpu[id] = a_gpu[id] + b_gpu[id];
        }
    }
    
""")

In [5]:
#################### Grid and Block size
block_size = 512
grid_size = int(np.ceil(N/block_size))


In [6]:
#################### Start GPU timing
start_gpu = cuda.Event()
end_gpu = cuda.Event()
start_gpu.record()

#################### Launch the GPU kernel
func = module.get_function("addition")
func(cuda.In(a_cpu), cuda.In(b_cpu), cuda.Out(c_cpu), np.uint32(N), grid=(grid_size, 1, 1), block=(block_size, 1, 1))

#################### End GPU timing
end_gpu.record()
cuda.Context.synchronize()
gpu_time = start_gpu.time_till(end_gpu)*1e-3
print("Elapsed on GPU with PyCuda (sec): ", gpu_time)
print("---------------------")

Elapsed on GPU with PyCuda (sec):  0.026879871368408203
---------------------


In [7]:
#################### Numba CPU parallel addition
@njit(parallel=True)
def cpu_addition(a, b, c):
    for i in prange(len(a)):
        c[i] = a[i] + b[i]

In [8]:
c_numba = np.zeros(N, np.uint32)
start_cpu = time.time()
cpu_addition(a_cpu, b_cpu, c_numba)
end_cpu = time.time()
cpu_time = end_cpu - start_cpu
print("Elapsed time on CPU using Numba (sec): ", cpu_time)
print("---------------------")

Elapsed time on CPU using Numba (sec):  0.672006368637085
---------------------


In [9]:
#################### Validation
dif = np.sum(c_cpu != c_numba)
print("Validation: there are %d different element(s)!" % dif)
print("---------------------")

Validation: there are 0 different element(s)!
---------------------
